In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from mpl_toolkits.mplot3d import Axes3D

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
%matplotlib inline
np.random.seed(42)

## 1. Generar y Explorar los Datos

In [ ]:
# Generar dataset sintético de clientes
n_clientes = 200

# Crear diferentes segmentos de clientes
# Segmento 1: Jóvenes con ingresos bajos pero alto gasto
edad_1 = np.random.normal(25, 5, 50)
ingreso_1 = np.random.normal(40, 10, 50)
gasto_1 = np.random.normal(75, 10, 50)

# Segmento 2: Adultos con ingresos medios y gasto moderado
edad_2 = np.random.normal(45, 8, 50)
ingreso_2 = np.random.normal(70, 15, 50)
gasto_2 = np.random.normal(50, 10, 50)

# Segmento 3: Adultos mayores con altos ingresos pero bajo gasto
edad_3 = np.random.normal(55, 7, 50)
ingreso_3 = np.random.normal(90, 12, 50)
gasto_3 = np.random.normal(30, 8, 50)

# Segmento 4: Jóvenes profesionales con altos ingresos y alto gasto
edad_4 = np.random.normal(35, 6, 50)
ingreso_4 = np.random.normal(85, 10, 50)
gasto_4 = np.random.normal(80, 8, 50)

# Combinar todos los segmentos
df = pd.DataFrame({
    'CustomerID': range(1, n_clientes + 1),
    'Age': np.concatenate([edad_1, edad_2, edad_3, edad_4]),
    'Annual_Income': np.concatenate([ingreso_1, ingreso_2, ingreso_3, ingreso_4]),
    'Spending_Score': np.concatenate([gasto_1, gasto_2, gasto_3, gasto_4])
})

# Asegurar valores positivos
df['Age'] = df['Age'].clip(lower=18, upper=70)
df['Annual_Income'] = df['Annual_Income'].clip(lower=15, upper=140)
df['Spending_Score'] = df['Spending_Score'].clip(lower=1, upper=99)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

In [ ]:
# Descripción de las características
print("Descripción de las variables:\n")
print("CustomerID: Identificador único del cliente")
print("Age: Edad del cliente (años)")
print("Annual_Income: Ingreso anual del cliente (miles de $)")
print("Spending_Score: Puntuación de gasto (1-100, basada en comportamiento)")

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
df.describe()

In [ ]:
# Verificar valores nulos
print("Valores nulos:")
print(df.isnull().sum())

In [ ]:
# Visualizar distribuciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Age'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de Edad')

axes[1].hist(df['Annual_Income'], bins=20, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Ingreso Anual (k$)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Ingreso')

axes[2].hist(df['Spending_Score'], bins=20, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_xlabel('Puntuación de Gasto')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Gasto')

plt.tight_layout()
plt.show()

In [ ]:
# Visualización de pares de variables
sns.pairplot(df[['Age', 'Annual_Income', 'Spending_Score']], diag_kind='kde')
plt.suptitle('Relaciones entre Variables', y=1.01)
plt.tight_layout()
plt.show()

## 2. Preparar los Datos

In [ ]:
# Seleccionar características para clustering
X = df[['Age', 'Annual_Income', 'Spending_Score']].values

print(f"Shape de los datos: {X.shape}")
print(f"Total de clientes: {X.shape[0]}")
print(f"Número de características: {X.shape[1]}")

In [ ]:
# Normalizar los datos (importante para K-means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Datos normalizados exitosamente")
print(f"Media de X_scaled: {X_scaled.mean(axis=0)}")
print(f"Desviación estándar de X_scaled: {X_scaled.std(axis=0)}")

## 3. Determinar el Número Óptimo de Clusters

In [ ]:
# Método del codo (Elbow Method)
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualizar método del codo
plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, marker='o', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Inercia (Within-Cluster Sum of Squares)')
plt.title('Método del Codo para Determinar k Óptimo')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Busca el 'codo' en la gráfica donde la disminución de inercia se hace más lenta")

In [ ]:
# Método de la Silueta (Silhouette Score)
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

# Visualizar silhouette scores
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, marker='s', linewidth=2, markersize=8, color='green')
plt.xlabel('Número de Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score para Diferentes Valores de k')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Encontrar el k óptimo
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\nNúmero óptimo de clusters según Silhouette Score: {optimal_k}")
print(f"Mejor Silhouette Score: {max(silhouette_scores):.4f}")

## 4. Entrenar el Modelo K-means

In [ ]:
# Entrenar K-means con el número óptimo de clusters
n_clusters = 4  # Basado en el análisis anterior

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, max_iter=300)
clusters = kmeans.fit_predict(X_scaled)

print(f"Modelo K-means entrenado con {n_clusters} clusters")
print(f"\nNúmero de iteraciones: {kmeans.n_iter_}")
print(f"Inercia final: {kmeans.inertia_:.2f}")

In [ ]:
# Agregar las etiquetas de cluster al DataFrame
df['Cluster'] = clusters

# Distribución de clientes por cluster
print("Distribución de clientes por cluster:")
print(df['Cluster'].value_counts().sort_index())

# Visualización
plt.figure(figsize=(8, 5))
df['Cluster'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Cluster')
plt.ylabel('Número de Clientes')
plt.title('Distribución de Clientes por Cluster')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Evaluar el Modelo

In [ ]:
# Calcular métricas de evaluación
silhouette = silhouette_score(X_scaled, clusters)
davies_bouldin = davies_bouldin_score(X_scaled, clusters)

print("Métricas de Evaluación:")
print(f"Silhouette Score: {silhouette:.4f} (rango: -1 a 1, mejor cerca de 1)")
print(f"Davies-Bouldin Index: {davies_bouldin:.4f} (menor es mejor)")
print(f"Inercia: {kmeans.inertia_:.2f}")

## 6. Visualizar los Clusters

In [ ]:
# Visualización 2D: Ingreso vs Gasto
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(df['Annual_Income'], df['Spending_Score'], 
                     c=df['Cluster'], cmap='viridis', 
                     s=100, alpha=0.6, edgecolors='k')

# Añadir centroides (desnormalizados)
centroides = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroides[:, 1], centroides[:, 2], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Centroides')

plt.xlabel('Ingreso Anual (k$)')
plt.ylabel('Puntuación de Gasto')
plt.title('Clusters: Ingreso vs Gasto')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
scatter = plt.scatter(df['Age'], df['Spending_Score'], 
                     c=df['Cluster'], cmap='viridis', 
                     s=100, alpha=0.6, edgecolors='k')

plt.scatter(centroides[:, 0], centroides[:, 2], 
           c='red', marker='X', s=300, edgecolors='black', linewidths=2,
           label='Centroides')

plt.xlabel('Edad (años)')
plt.ylabel('Puntuación de Gasto')
plt.title('Clusters: Edad vs Gasto')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 3D
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(df['Age'], df['Annual_Income'], df['Spending_Score'],
                    c=df['Cluster'], cmap='viridis', 
                    s=100, alpha=0.6, edgecolors='k')

# Añadir centroides
ax.scatter(centroides[:, 0], centroides[:, 1], centroides[:, 2],
          c='red', marker='X', s=300, edgecolors='black', linewidths=2,
          label='Centroides')

ax.set_xlabel('Edad')
ax.set_ylabel('Ingreso Anual (k$)')
ax.set_zlabel('Puntuación de Gasto')
ax.set_title('Visualización 3D de Clusters de Clientes')
plt.colorbar(scatter, label='Cluster')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Análisis de Perfiles de Clusters

In [ ]:
# Estadísticas por cluster
cluster_stats = df.groupby('Cluster')[['Age', 'Annual_Income', 'Spending_Score']].mean()

print("Características promedio por cluster:")
print(cluster_stats.round(2))

# Añadir conteo
cluster_stats['Count'] = df.groupby('Cluster').size()
print("\nTabla completa con conteos:")
print(cluster_stats)

In [ ]:
# Visualizar perfiles de clusters
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cluster_means = df.groupby('Cluster')[['Age', 'Annual_Income', 'Spending_Score']].mean()

cluster_means['Age'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Edad Promedio')
axes[0].set_title('Edad Promedio por Cluster')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

cluster_means['Annual_Income'].plot(kind='bar', ax=axes[1], color='green', edgecolor='black')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Ingreso Promedio (k$)')
axes[1].set_title('Ingreso Promedio por Cluster')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

cluster_means['Spending_Score'].plot(kind='bar', ax=axes[2], color='coral', edgecolor='black')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Gasto Promedio')
axes[2].set_title('Gasto Promedio por Cluster')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Interpretación de clusters
print("\nINTERPRETACIÓN DE CLUSTERS:\n")
for i in range(n_clusters):
    cluster_data = df[df['Cluster'] == i]
    print(f"Cluster {i}:")
    print(f"  - Tamaño: {len(cluster_data)} clientes ({len(cluster_data)/len(df)*100:.1f}%)")
    print(f"  - Edad promedio: {cluster_data['Age'].mean():.1f} años")
    print(f"  - Ingreso promedio: ${cluster_data['Annual_Income'].mean():.1f}k")
    print(f"  - Gasto promedio: {cluster_data['Spending_Score'].mean():.1f}/100")
    print()

## 8. Predecir Cluster para Nuevos Clientes

In [ ]:
# Ejemplo: clasificar nuevos clientes
nuevos_clientes = pd.DataFrame({
    'Age': [28, 50, 35],
    'Annual_Income': [45, 85, 70],
    'Spending_Score': [80, 25, 55]
}, index=['Cliente A', 'Cliente B', 'Cliente C'])

print("Nuevos clientes a clasificar:")
print(nuevos_clientes)

# Normalizar y predecir
nuevos_clientes_scaled = scaler.transform(nuevos_clientes)
clusters_predichos = kmeans.predict(nuevos_clientes_scaled)

nuevos_clientes['Cluster_Asignado'] = clusters_predichos

print("\nClusters asignados:")
print(nuevos_clientes)

## Conclusiones

- K-means es un algoritmo eficiente para segmentar datos en grupos homogéneos
- El número óptimo de clusters debe determinarse usando métodos como el codo y Silhouette Score
- La normalización de datos es crucial para K-means ya que es sensible a la escala
- Los centroides representan el "cliente promedio" de cada segmento
- Es útil para marketing, segmentación de clientes y detección de patrones en datos sin etiquetar